# 验证与泛化

## 学习目标

划分训练与验证集，观察过拟合，并使用 L2、Dropout 和早停思想控制模型复杂度。

## 概念模型

训练集用于更新参数，验证集用于选择超参数和停止时机。正则化只影响训练规则，验证数据绝不能参与 backward。

## 逐步实现

按顺序运行下面的代码，并在每一步检查 shape、数值范围和中间结果。

In [ ]:
from pathlib import Path
import sys

course_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd() / "07-deep-learning/fundamentals"
sys.path.insert(0, str(course_dir.resolve()))

import numpy as np
from from_scratch import CrossEntropyLoss, MLP, Momentum, accuracy, make_spiral, train_classifier, train_validation_split

x, y = make_spiral(samples_per_class=80, noise=0.22, seed=12)
x_train, x_valid, y_train, y_valid = train_validation_split(x, y, seed=12)
model = MLP(2, [48, 48], 3, dropout=0.1, seed=12)
optimizer = Momentum(model.parameters(), learning_rate=0.07, momentum=0.9, weight_decay=1e-4)
history = train_classifier(model, x_train, y_train, optimizer, epochs=140, batch_size=32, seed=12)
valid_logits = model.forward(x_valid, training=False)
valid_loss = CrossEntropyLoss().forward(valid_logits, y_valid)
print("train accuracy:", history["accuracy"][-1])
print("validation loss/accuracy:", valid_loss, accuracy(valid_logits.argmax(1), y_valid))

In [ ]:
# 早停的核心状态：只在验证损失改善时保存参数副本。
best_loss = float("inf")
best_parameters = None
if valid_loss < best_loss:
    best_loss = valid_loss
    best_parameters = [parameter.copy() for parameter, _ in model.parameters()]
print("saved tensors:", len(best_parameters))

## 检查点

Dropout 在训练时随机丢弃激活并做缩放，验证时必须关闭。L2 通过 weight_decay 惩罚过大的参数。

## 试一试

改变隐藏层宽度和 Dropout 概率，比较训练与验证准确率之间的差距。

## 常见错误

用验证集更新参数；验证时仍打开 Dropout；只保存模型对象引用而不是最佳参数副本。